# Thí nghiệm Giai đoạn 1 — Kiểm tra dữ liệu và chuẩn hóa đánh giá Wi-Fi Fingerprint

**Đề tài:** Cải thiện chất lượng Wi-Fi Fingerprint sinh bởi Conditional GAN và đánh giá
hiệu quả tăng cường dữ liệu cho định vị tại vị trí chưa khảo sát.

Notebook này là bản thử nghiệm độc lập từ `KLTN_CGAN.ipynb`.
**Không sửa notebook gốc, không thay Generator/Discriminator, chưa huấn luyện cGAN.**
Chỉ chuẩn bị quy trình kiểm tra và baseline **dữ liệu thật + KNN/WKNN**.
Không chứa kết quả thực nghiệm được tạo sẵn hoặc dữ liệu giả thay cho dataset.

Chạy lần lượt từ trên xuống trong runtime Colab mới; CPU là đủ.
Nếu thiếu ZIP, chọn tải lên hoặc Google Drive ở mục 4.
Test mặc định chưa chạy: chỉ bật `RUN_FINAL_TEST` ngay mục 13 sau khi khóa cấu hình bằng Validation.

**Phạm vi:** mô phỏng vị trí chưa khảo sát bằng cách giữ lại toàn bộ mẫu của RP.
Test nội bộ được tách từ các file **Train**, không tự thay bằng các file `DLULIB_Test_*`.
Test nội bộ này đã từng xuất hiện trong notebook cũ; không tuyên bố đó là holdout chưa được quan sát.
Định vị riêng từng tầng, giả định đã biết tầng và X/Y có đơn vị mét.


## Những phần được chuẩn hóa từ notebook gốc

| Cell gốc | Chức năng trong bản thử nghiệm |
|---|---|
| 1.1–1.3; 2.2–2.3 | Đọc sheet `_2`; audit RSSI, kiểu dữ liệu, RP và tọa độ |
| 2.1; 3.10; 4.3.13D | Tách Train/Validation/Test theo RP, kiểm tra giao nhau, lưu manifest |
| 4.3.3/4.3.3B | Baseline KNN (`uniform`) và WKNN (`distance`) chỉ từ dữ liệu thật |
| 4.3.14–4.3.16 | Một định nghĩa khoảng cách; chọn k trên Validation; Test sau khóa cấu hình |
| Dashboard cuối | Bảng, biểu đồ và hồ sơ thí nghiệm có thể xuất ra tệp |

Không chép các output cũ sang notebook này. Không xóa dòng lặp, tự điền NaN,
hoặc sửa Excel gốc. Các chính sách chuyển đổi phải được cấu hình và ghi lại.


## 1. Chuẩn bị thư viện

Chỉ cài thư viện nếu chưa có, dùng PyPI với kiểm tra TLS mặc định.
Ghi phiên bản thực tế vào hồ sơ thí nghiệm. Không cần GPU hoặc PyTorch ở giai đoạn này.


In [ ]:
import importlib.util
import subprocess
import sys

requirements = {
    "numpy": "numpy>=1.24,<3", "pandas": "pandas>=2,<3",
    "sklearn": "scikit-learn>=1.3,<2", "matplotlib": "matplotlib>=3.7,<4",
    "openpyxl": "openpyxl>=3.1,<4",
}
missing = [package for module, package in requirements.items()
           if importlib.util.find_spec(module) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", *missing])

import copy
import hashlib
import io
import json
import os
import shutil
import stat
import tempfile
import uuid
import warnings
import zipfile
from datetime import datetime, timezone
from importlib.metadata import version
from itertools import combinations
from pathlib import Path, PurePosixPath

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from threadpoolctl import threadpool_limits

LIBRARY_VERSIONS = {name: version(name) for name in
                    ("numpy", "pandas", "scikit-learn", "matplotlib", "openpyxl")}
display(pd.Series(LIBRARY_VERSIONS, name="Phiên bản thực tế"))


## 2. Cấu hình thí nghiệm

- Mặc định `k_candidates=[5]`: giữ k=5 để đối chiếu phương pháp cũ.
  Nếu đổi thành `[3, 5, 7, 9]`, k được chọn **chỉ trên Validation**, riêng mỗi tầng/phương pháp.
- `100` mặc định gây lỗi. Chỉ chọn `map_to_missing` khi đã xác minh `100` là mã không phát hiện.
  Việc chuyển đổi chỉ thực hiện trên bản dữ liệu trong RAM và được đếm lại.
- KNN dùng dBm và StandardScaler. **Không clip RSSI về [-100, -20].**
  Giá trị lớn hơn -20 được báo riêng vì không nhất thiết sai vật lý.
- Công tắc Test nằm ngay mục 13 để không khởi tạo lại thư mục lần chạy.
  Không cần chạy lại cell cấu hình này chỉ để bật Test.


In [ ]:
CONFIG = {
    "seed": 42,
    "test_fraction": 0.20,
    "validation_fraction_of_remaining": 0.15,
    "floors": ["F1", "F2", "F3"],
    "missing_rssi_dbm": -100.0,
    "reference_gan_max_dbm": -20.0,
    "sentinel_100_policy": "error",  # "error" hoặc "map_to_missing"
    "coordinate_decimals": 4,       # chỉ dùng tạo khóa kiểm tra vị trí
    "coordinate_unit": "m",         # cần đối chiếu tài liệu dataset
    "expected_samples_per_rp": 20,  # sai khác chỉ cảnh báo, không xóa mẫu
    "k_candidates": [5],
    "methods": {"KNN": "uniform", "WKNN": "distance"},
    "selection_metric": "MAE_distance_m",
    "refit_train_and_validation": True,
    "n_jobs": 2,
}

DATA_SOURCE = "auto"  # "auto", "upload", "drive", "path"
ZIP_PATH = ""         # đường dẫn ZIP khi chọn "path", hoặc gợi ý cho "auto"
DRIVE_ZIP_PATH = "/content/drive/MyDrive/DATASET_DLULIB.zip"

AP_COLS = [f"AP{i}" for i in range(1, 18)]
XY_COLS = ["X", "Y"]
REQUIRED_COLS = ["ID_RP", *XY_COLS, *AP_COLS]
if not all(0 < CONFIG[key] < 1 for key in
           ("test_fraction", "validation_fraction_of_remaining")):
    raise ValueError("Tỷ lệ chia tập phải nằm trong (0, 1).")
if CONFIG["sentinel_100_policy"] not in {"error", "map_to_missing"}:
    raise ValueError("Chính sách sentinel không hợp lệ.")
if not CONFIG["k_candidates"] or any(
        isinstance(k, bool) or not isinstance(k, int) or k < 1
        for k in CONFIG["k_candidates"]):
    raise ValueError("k_candidates phải là danh sách số nguyên dương.")
if CONFIG["methods"] != {"KNN": "uniform", "WKNN": "distance"}:
    raise ValueError("Giai đoạn 1 đăng ký trước hai baseline KNN và WKNN.")
if CONFIG["coordinate_unit"] != "m":
    raise ValueError("Hãy đổi tọa độ sang mét theo tài liệu trước khi dùng metric mét.")

default_root = (Path("/content/wifi_fingerprint_stage1") if Path("/content").is_dir()
                else Path(tempfile.gettempdir()) / "wifi_fingerprint_stage1")
RUNTIME_ROOT = Path(os.environ.get("WIFI_EXPERIMENT_DIR", str(default_root)))
RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "_" + uuid.uuid4().hex[:8]
OUTPUT_DIR = RUNTIME_ROOT / "runs" / RUN_ID
OUTPUT_DIR.mkdir(parents=True, exist_ok=False)
print("Thư mục kết quả của lần chạy:", OUTPUT_DIR)
print("Test mặc định chưa bật; công tắc riêng ở mục 13.")


## 3. Công thức đánh giá và tự kiểm tra toán học

Với mỗi mẫu, khoảng cách định vị là
\(e_i=\sqrt{(\hat x_i-x_i)^2+(\hat y_i-y_i)^2}\).

- `MAE_distance_m` (MDE): \(\frac{1}{N}\sum e_i\).
- `RMSE_distance_m`: \(\sqrt{\frac{1}{N}\sum e_i^2}\).
- `MAE_X_m`, `MAE_Y_m`: chỉ số riêng từng trục, không thay thế khoảng cách.
- `MAE_RP_macro_m`: trung bình lỗi theo RP rồi lấy trung bình các RP,
  tránh RP nhiều mẫu có trọng số lớn hơn mà không được nhận biết.

Hàm `rssi_metrics` chuẩn bị cho giai đoạn có fingerprint đối chứng:
MAE/RMSE của chênh lệch RSSI dBm có đơn vị dB. Ở giai đoạn này **chưa có mô hình tái tạo RSSI**,
nên không xuất bảng chất lượng RSSI giả định.

Ví dụ 3–4–5 bên dưới chỉ kiểm tra phần mềm, không phải kết quả dataset.


In [ ]:
def paired_arrays(y_true, y_pred, n_columns=None):
    true = np.asarray(y_true, dtype=np.float64)
    pred = np.asarray(y_pred, dtype=np.float64)
    if true.ndim != 2 or true.shape != pred.shape or true.shape[0] == 0:
        raise ValueError("Hai mảng phải cùng shape (N, D), N > 0.")
    if n_columns is not None and true.shape[1] != n_columns:
        raise ValueError(f"Cần đúng {n_columns} cột.")
    if true.shape[1] == 0 or not (np.isfinite(true).all() and np.isfinite(pred).all()):
        raise ValueError("Đầu vào metric rỗng hoặc có NaN/Inf.")
    return true, pred


def localization_metrics(y_true, y_pred, rp_keys=None):
    true, pred = paired_arrays(y_true, y_pred, n_columns=2)
    delta = pred - true
    errors = np.linalg.norm(delta, axis=1)
    metrics = {
        "N_samples": len(errors),
        "MAE_distance_m": float(errors.mean()),
        "RMSE_distance_m": float(np.sqrt(np.mean(errors ** 2))),
        "Median_distance_m": float(np.median(errors)),
        "P90_distance_m": float(np.quantile(errors, 0.90)),
        "MAE_X_m": float(np.abs(delta[:, 0]).mean()),
        "MAE_Y_m": float(np.abs(delta[:, 1]).mean()),
    }
    if rp_keys is not None:
        keys = np.asarray(rp_keys)
        if keys.ndim != 1 or len(keys) != len(errors) or pd.isna(keys).any():
            raise ValueError("Khóa RP phải đầy đủ và khớp số mẫu.")
        per_rp = pd.DataFrame({"RP_key": keys, "error_m": errors}).groupby("RP_key")["error_m"].mean()
        metrics.update(N_RP=len(per_rp), MAE_RP_macro_m=float(per_rp.mean()))
    return metrics, errors


def rssi_metrics(real_dbm, predicted_dbm, evaluation_mask=None):
    # Mask đánh giá phải dựa trên giao thức/ground truth, không lọc bằng dự đoán.
    true, pred = paired_arrays(real_dbm, predicted_dbm)
    if evaluation_mask is None:
        mask = np.ones(true.shape, dtype=bool)
    else:
        mask = np.asarray(evaluation_mask)
        if mask.shape != true.shape or mask.dtype != bool:
            raise ValueError("evaluation_mask phải là bool và cùng shape RSSI.")
    if not mask.any():
        raise ValueError("Không có ô RSSI hợp lệ để tính metric.")
    diff = (pred - true)[mask]
    return {"N_RSSI_cells": int(mask.sum()),
            "RSSI_MAE_dB": float(np.abs(diff).mean()),
            "RSSI_RMSE_dB": float(np.sqrt(np.mean(diff ** 2)))}


# Kiểm tra công thức, không tạo dataset thực nghiệm.
check, _ = localization_metrics([[0, 0]], [[3, 4]], ["check"])
assert check["MAE_distance_m"] == check["RMSE_distance_m"] == 5.0
assert check["MAE_X_m"] == 3.0 and check["MAE_Y_m"] == 4.0
check, _ = localization_metrics([[0, 0], [0, 0]], [[0, 0], [3, 4]])
assert np.isclose(check["MAE_distance_m"], 2.5)
assert np.isclose(check["RMSE_distance_m"], np.sqrt(12.5))
assert rssi_metrics([[-60, -70]], [[-63, -66]])["RSSI_MAE_dB"] == 3.5
print("PASS: kiểm tra toán học metric. Chưa phải kết quả dữ liệu thật.")


## 4. Chọn DATASET_DLULIB.zip: tải lên hoặc Google Drive

`auto` tìm ZIP ở đường dẫn cấu hình, `/content` và thư mục làm việc. Nếu chưa có,
Colab hỏi chọn **upload** hoặc **drive**. Với Drive, đặt `DRIVE_ZIP_PATH` ở mục 2.
Ngoài Colab, dùng `DATA_SOURCE='path'` và `ZIP_PATH`.

Chấp nhận ZIP có thư mục lồng. Ba file `DLULIB_Train_F*.xlsx` là bắt buộc.
Các file `DLULIB_Test_*` chỉ được kiểm kê và **không đưa vào Test nội bộ**;
chúng có thể thiếu X/Y và cần giao thức riêng trước khi dùng.
Chỉ sao chép workbook Train cần dùng vào thư mục chạy, không sửa ZIP/Excel nguồn.


In [ ]:
def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def resolve_zip(source, zip_path="", drive_zip_path=""):
    if source not in {"auto", "upload", "drive", "path"}:
        raise ValueError("DATA_SOURCE không hợp lệ.")
    if source == "path":
        candidate = Path(zip_path).expanduser()
        if not zip_path or not candidate.is_file():
            raise FileNotFoundError("Đặt ZIP_PATH tới file ZIP có thật.")
        return candidate.resolve()
    if source == "auto":
        candidates = ([Path(zip_path).expanduser()] if zip_path else [])
        candidates += [Path("/content/DATASET_DLULIB.zip"), Path.cwd() / "DATASET_DLULIB.zip"]
        for candidate in candidates:
            if candidate.is_file():
                return candidate.resolve()
        try:
            import google.colab  # chỉ kiểm tra có đang ở Colab không
        except ImportError as exc:
            raise FileNotFoundError(
                "Chưa có dữ liệu thật. Dùng DATA_SOURCE='path' và đặt ZIP_PATH; "
                "hoặc mở notebook trong Colab để upload/Drive. Không tạo dữ liệu thay thế."
            ) from exc
        answer = input("Chưa có ZIP. Nhập upload hoặc drive [upload]: ").strip().lower()
        source = answer or "upload"
        if source not in {"upload", "drive"}:
            raise ValueError("Chỉ chọn upload hoặc drive.")
    if source == "drive":
        try:
            from google.colab import drive
        except ImportError as exc:
            raise RuntimeError("Mount Drive chỉ dùng trên Google Colab.") from exc
        drive.mount("/content/drive")
        candidate = Path(drive_zip_path).expanduser()
        if not candidate.is_file():
            raise FileNotFoundError(f"Không thấy ZIP trên Drive: {candidate}")
        return candidate.resolve()
    try:
        from google.colab import files
    except ImportError as exc:
        raise RuntimeError("Upload tương tác chỉ dùng trên Colab; hãy dùng đường dẫn local.") from exc
    upload_dir = OUTPUT_DIR / "uploaded_input"
    upload_dir.mkdir(exist_ok=True)
    previous = Path.cwd()
    try:
        os.chdir(upload_dir)
        uploaded = files.upload()
    finally:
        os.chdir(previous)
    names = [name for name in uploaded if name.lower().endswith(".zip")]
    if len(names) != 1:
        raise ValueError("Hãy tải lên đúng một file ZIP dữ liệu.")
    candidate = (upload_dir / names[0]).resolve()
    if not candidate.is_relative_to(upload_dir.resolve()):
        raise ValueError("Tên file tải lên không hợp lệ.")
    return candidate


def prepare_workbooks(zip_path, target_dir):
    required = [f"DLULIB_Train_{floor}.xlsx" for floor in CONFIG["floors"]]
    optional = [f"DLULIB_Test_{floor}.xlsx" for floor in CONFIG["floors"]]
    expected = {name.casefold(): name for name in required + optional}
    matches = {name: [] for name in expected}
    with zipfile.ZipFile(zip_path) as archive:
        members = archive.infolist()
        if sum(info.file_size for info in members) > 512 * 1024 ** 2:
            raise ValueError("ZIP có dung lượng giải nén >512 MiB; kiểm tra đúng dataset trước.")
        for info in members:
            path = PurePosixPath(info.filename.replace("\\", "/"))
            if path.is_absolute() or ".." in path.parts or stat.S_ISLNK(info.external_attr >> 16):
                raise ValueError(f"ZIP chứa đường dẫn không hợp lệ: {info.filename}")
            name = path.name.casefold()
            if not info.is_dir() and name in matches:
                matches[name].append(info)
        absent = [name for name in required if not matches[name.casefold()]]
        ambiguous = [expected[name] for name, items in matches.items() if len(items) > 1]
        if absent or ambiguous:
            raise ValueError(f"Thiếu workbook Train: {absent}; tên workbook không duy nhất: {ambiguous}")
        target_dir = Path(target_dir)
        target_dir.mkdir(parents=True, exist_ok=False)
        files = {}
        for name in required:
            destination = target_dir / name
            with archive.open(matches[name.casefold()][0]) as src, destination.open("xb") as dst:
                shutil.copyfileobj(src, dst)
            files[name] = destination
        inventory = pd.DataFrame([
            {"Workbook": name, "Có trong ZIP": bool(matches[name.casefold()]),
             "Vai trò": "Nguồn để chia RP nội bộ" if name in required else "Chỉ kiểm kê; không sử dụng"}
            for name in required + optional
        ])
    return files, inventory


DATA_ZIP = resolve_zip(DATA_SOURCE, ZIP_PATH, DRIVE_ZIP_PATH)
ZIP_SHA256 = sha256_file(DATA_ZIP)
input_dir = OUTPUT_DIR / "input_workbooks" / (ZIP_SHA256[:12] + "_" + uuid.uuid4().hex[:8])
WORKBOOK_FILES, workbook_inventory = prepare_workbooks(DATA_ZIP, input_dir)
print("ZIP nguồn:", DATA_ZIP)
print("SHA256:", ZIP_SHA256)
display(workbook_inventory)


## 5. Kiểm tra schema, RSSI, RP và tọa độ

Chặn thiếu cột, cột trùng, ID không hợp lệ, NaN/Inf, RSSI ngoài [-100, 0),
một RP có nhiều tọa độ hoặc nhiều ID cùng vị trí làm mất tính độc lập của split.
ID số trùng giữa các tầng là hợp lệ: khóa đầy đủ luôn có tầng.

Dòng đo lặp và vector RSSI giống nhau chỉ được thống kê, không tự xóa hoặc kết luận rò rỉ.
Số mẫu khác 20/RP và fingerprint toàn -100 được cảnh báo. Không chỉnh dữ liệu nguồn.


In [ ]:
def coordinate_keys(frame):
    rounded = frame[XY_COLS].round(CONFIG["coordinate_decimals"])
    return set(zip(frame["Floor"].astype(str), rounded["X"], rounded["Y"]))


def audit_frame(raw, floor):
    frame = raw.copy(deep=True)
    frame.columns = [str(column).strip() for column in frame.columns]
    if frame.columns.duplicated().any():
        raise ValueError(f"{floor}: cột trùng sau chuẩn hóa tên.")
    missing = set(REQUIRED_COLS) - set(frame.columns)
    if missing or frame.empty:
        raise ValueError(f"{floor}: thiếu cột {sorted(missing)} hoặc sheet rỗng.")
    extra = [column for column in frame.columns if column not in REQUIRED_COLS]
    frame = frame[REQUIRED_COLS].copy()
    frame[REQUIRED_COLS] = frame[REQUIRED_COLS].apply(pd.to_numeric, errors="coerce")
    numeric = frame[REQUIRED_COLS].to_numpy(dtype=float)
    if not np.isfinite(numeric).all():
        rows = np.flatnonzero(~np.isfinite(numeric).all(axis=1))[:10] + 2
        raise ValueError(f"{floor}: NaN/Inf/chuỗi không phải số tại dòng Excel {rows.tolist()}.")
    ids = frame["ID_RP"].to_numpy(dtype=float)
    if not np.equal(ids, np.floor(ids)).all() or np.abs(ids).max() > 2 ** 53:
        raise ValueError(f"{floor}: ID_RP phải là số nguyên có thể biểu diễn chính xác.")
    frame["ID_RP"] = frame["ID_RP"].astype(np.int64)
    values = frame[AP_COLS].to_numpy(dtype=float)
    n_100 = int((values == 100).sum())
    if n_100 and CONFIG["sentinel_100_policy"] == "error":
        raise ValueError(f"{floor}: có {n_100} ô bằng 100. Xác minh sentinel trước khi chọn map_to_missing.")
    if n_100:
        frame[AP_COLS] = frame[AP_COLS].replace(100, CONFIG["missing_rssi_dbm"])
    values = frame[AP_COLS].to_numpy(dtype=float)
    invalid = (values < CONFIG["missing_rssi_dbm"]) | (values >= 0)
    if invalid.any():
        raise ValueError(f"{floor}: {int(invalid.sum())} ô RSSI ngoài miền cho phép [-100, 0).")
    frame["Floor"] = floor
    frame["SourceRow"] = np.arange(len(frame), dtype=int) + 2
    frame["Row_key"] = floor + ":row:" + frame["SourceRow"].astype(str)
    frame["RP_key"] = floor + ":rp:" + frame["ID_RP"].astype(str)
    rounded = frame[XY_COLS].round(CONFIG["coordinate_decimals"])
    positions = pd.concat([frame[["ID_RP"]], rounded], axis=1).drop_duplicates()
    if positions.groupby("ID_RP").size().gt(1).any():
        raise ValueError(f"{floor}: một ID_RP có nhiều tọa độ; cần kiểm tra dữ liệu gốc.")
    if positions.duplicated(XY_COLS).any():
        raise ValueError(f"{floor}: nhiều ID_RP cùng vị trí; cần giao thức nhóm vị trí trước khi chia.")
    counts = frame.groupby("ID_RP").size()
    detected = values > CONFIG["missing_rssi_dbm"]
    report = {
        "Floor": floor, "N_samples": len(frame), "N_RP": len(counts),
        "Samples_RP_min": int(counts.min()), "Samples_RP_max": int(counts.max()),
        "RP_not_20_samples": int((counts != CONFIG["expected_samples_per_rp"]).sum()),
        "Duplicate_measurement_rows": int(frame.duplicated(REQUIRED_COLS).sum()),
        "All_missing_fingerprints": int((~detected.any(axis=1)).sum()),
        "Sentinel_100_mapped": n_100, "RSSI_min_dBm": float(values.min()),
        "RSSI_max_dBm": float(values.max()),
        "Cells_above_minus20": int((values > CONFIG["reference_gan_max_dbm"]).sum()),
        "Detection_percent": float(detected.mean() * 100), "Ignored_extra_columns": ", ".join(extra),
    }
    return frame, report


data_by_floor, audit_rows, read_errors = {}, [], []
for floor in CONFIG["floors"]:
    workbook = WORKBOOK_FILES[f"DLULIB_Train_{floor}.xlsx"]
    sheet = f"Train_{floor}_2"
    try:
        with pd.ExcelFile(workbook, engine="openpyxl") as excel:
            if sheet not in excel.sheet_names:
                raise ValueError(f"Thiếu sheet {sheet}; có: {excel.sheet_names}")
            raw = pd.read_excel(excel, sheet_name=sheet)
        data_by_floor[floor], report = audit_frame(raw, floor)
        audit_rows.append(report)
    except (ValueError, OSError, KeyError) as exc:
        read_errors.append({"Floor": floor, "Lỗi": str(exc)})
audit_df = pd.DataFrame(audit_rows)
if len(audit_df):
    display(audit_df)
if read_errors:
    display(pd.DataFrame(read_errors))
    raise ValueError("Dừng trước huấn luyện: hãy xử lý/xác minh các lỗi đầu vào được liệt kê.")
if audit_df[["RP_not_20_samples", "Duplicate_measurement_rows", "All_missing_fingerprints",
             "Cells_above_minus20"]].to_numpy().any():
    warnings.warn("Có cảnh báo dữ liệu trong bảng audit; không tự xóa, điền hoặc clip giá trị.")
print("Đã kiểm tra dữ liệu trong RAM; Excel và ZIP nguồn được giữ nguyên.")


## 6. Chia Train, Validation, Test theo RP

Giữ thuật toán của notebook gốc: tách 20% RP làm Test, rồi 15% RP còn lại làm Validation.
Xấp xỉ 68%/12%/20%, có sai khác do làm tròn. Outer split giữ thứ tự ID xuất hiện;
inner split sắp xếp ID như Cell 3.10 gốc. Không cam kết trùng split/output cũ khi dữ liệu thay đổi.
Mọi mẫu của cùng RP đi cùng một tập. Không dùng RSSI để quyết định chia tập.
Đây là split RP ngẫu nhiên, chưa phải thí nghiệm giữ lại cả vùng không gian.


In [ ]:
def split_floor_by_rp(frame):
    rp_ids = frame["ID_RP"].unique()
    try:
        remaining_ids, test_ids = train_test_split(
            rp_ids, test_size=CONFIG["test_fraction"], random_state=CONFIG["seed"])
        train_ids, val_ids = train_test_split(
            np.sort(remaining_ids),
            test_size=CONFIG["validation_fraction_of_remaining"],
            random_state=CONFIG["seed"])
    except ValueError as exc:
        raise ValueError("Không đủ RP để tạo ba tập không rỗng theo tỷ lệ cấu hình.") from exc
    id_sets = {"Train": set(train_ids), "Validation": set(val_ids), "Test": set(test_ids)}
    return {name: frame.loc[frame["ID_RP"].isin(ids)].copy().reset_index(drop=True)
            for name, ids in id_sets.items()}


splits = {floor: split_floor_by_rp(frame) for floor, frame in data_by_floor.items()}
split_rows, manifest_parts = [], []
for floor, parts in splits.items():
    for name, frame in parts.items():
        split_rows.append({"Floor": floor, "Split": name, "N_samples": len(frame),
                           "N_RP": frame["RP_key"].nunique()})
        manifest_parts.append(frame[["Floor", "ID_RP", "RP_key", "Row_key", "SourceRow", *XY_COLS]]
                              .assign(Split=name))
split_summary_df = pd.DataFrame(split_rows)
split_manifest_df = pd.concat(manifest_parts, ignore_index=True)
display(split_summary_df)


## 7. Kiểm tra độc lập và bảo toàn mẫu

Kiểm tra cả khóa RP, tọa độ có tầng và dòng nguồn giữa từng cặp tập.
Vector RSSI giống nhau giữa các tập được thống kê riêng: đó **không tự động là rò rỉ**,
vì nhiều RP có thể thu được tín hiệu giống nhau, nhất là khi nhiều AP không phát hiện.
Không có metadata phiên thu thập thì chưa thể xác minh độc lập theo phiên/thời gian/thiết bị.


In [ ]:
def verify_partitions(source, parts):
    names = ["Train", "Validation", "Test"]
    if set(parts) != set(names) or any(parts[name].empty for name in names):
        raise ValueError("Cần đúng ba tập không rỗng.")
    if sum(len(frame) for frame in parts.values()) != len(source):
        raise AssertionError("Không bảo toàn số dòng.")
    combined_rows = [key for frame in parts.values() for key in frame["Row_key"]]
    if len(set(combined_rows)) != len(combined_rows) or set(combined_rows) != set(source["Row_key"]):
        raise AssertionError("Có dòng bị mất, lặp hoặc không thuộc nguồn.")
    rows = []
    for left, right in combinations(names, 2):
        a, b = parts[left], parts[right]
        overlaps = {
            "RP_overlap": len(set(a["RP_key"]) & set(b["RP_key"])),
            "Coordinate_overlap": len(coordinate_keys(a) & coordinate_keys(b)),
            "Source_row_overlap": len(set(a["Row_key"]) & set(b["Row_key"])),
        }
        if any(overlaps.values()):
            raise AssertionError(f"Các tập giao nhau: {left}/{right}: {overlaps}")
        rss_a = set(pd.util.hash_pandas_object(a[AP_COLS], index=False))
        rss_b = set(pd.util.hash_pandas_object(b[AP_COLS], index=False))
        rows.append({"Pair": f"{left}/{right}", **overlaps,
                     "Identical_RSSI_vector_hashes": len(rss_a & rss_b)})
    return rows


overlap_rows = []
for floor, parts in splits.items():
    overlap_rows.extend({"Floor": floor, **row}
                        for row in verify_partitions(data_by_floor[floor], parts))
overlap_audit_df = pd.DataFrame(overlap_rows)
display(overlap_audit_df)
print("PASS: bảo toàn mẫu; không giao RP, vị trí làm tròn hoặc dòng nguồn giữa ba tập.")


## 8. Bản đồ vị trí của các tập

Bản đồ mô tả split, không phải kết quả định vị. Tọa độ Test chỉ hiển thị để kiểm tra giao thức;
không dùng để điều chỉnh mô hình. Chưa có mặt bằng để xác minh tường/vùng tiếp cận.


In [ ]:
fig, axes = plt.subplots(1, len(CONFIG["floors"]), figsize=(16, 4.5), squeeze=False)
colors = {"Train": "#2563eb", "Validation": "#d97706", "Test": "#7c3aed"}
for ax, floor in zip(axes[0], CONFIG["floors"]):
    for name, frame in splits[floor].items():
        rp = frame.drop_duplicates("RP_key")
        ax.scatter(rp.X, rp.Y, s=18, label=f"{name}: {len(rp)} RP", color=colors[name])
    ax.set(title=f"{floor} — chia theo RP", xlabel="X (m)", ylabel="Y (m)")
    ax.set_aspect("equal", adjustable="box")
    ax.grid(alpha=0.2)
    ax.legend(fontsize=8)
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "split_map.png", dpi=160, bbox_inches="tight")
plt.show()
plt.close(fig)


## 9. Khảo sát RSSI chỉ trên Train

Histogram tách RSSI có phát hiện khỏi mã -100. Biểu đồ detection thể hiện từng AP.
Thống kê AP hằng số giúp giải thích StandardScaler/KNN; không tự loại AP dựa trên Test.


In [ ]:
train_ap_rows = []
fig, axes = plt.subplots(len(CONFIG["floors"]), 2, figsize=(14, 10), squeeze=False)
for index, floor in enumerate(CONFIG["floors"]):
    frame = splits[floor]["Train"]
    values = frame[AP_COLS].to_numpy(dtype=float)
    detected = values > CONFIG["missing_rssi_dbm"]
    observed = values[detected]
    if observed.size:
        axes[index, 0].hist(observed, bins=35, color="#2563eb", alpha=0.8)
    else:
        axes[index, 0].text(0.5, 0.5, "Không có AP phát hiện", ha="center")
    axes[index, 0].set(title=f"{floor} — RSSI có phát hiện (Train)",
                       xlabel="RSSI (dBm)", ylabel="Số ô RSSI")
    axes[index, 1].bar(AP_COLS, detected.mean(axis=0) * 100, color="#0f766e")
    axes[index, 1].set(title=f"{floor} — tỷ lệ phát hiện (Train)",
                       ylabel="% mẫu", ylim=(0, 100))
    axes[index, 1].tick_params(axis="x", rotation=60)
    for j, ap in enumerate(AP_COLS):
        train_ap_rows.append({"Floor": floor, "AP": ap,
                              "Detection_percent": float(detected[:, j].mean() * 100),
                              "Train_std_dB": float(values[:, j].std()),
                              "Constant_on_train": bool(np.ptp(values[:, j]) == 0)})
train_ap_audit_df = pd.DataFrame(train_ap_rows)
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "train_rssi_audit.png", dpi=160, bbox_inches="tight")
plt.show()
plt.close(fig)
display(train_ap_audit_df)


## 10. Baseline KNN/WKNN và lựa chọn chỉ trên Validation

Đầu vào là 17 RSSI, đầu ra là X/Y theo mét. Pipeline:
`StandardScaler (fit Train) → KNeighborsRegressor`.

- KNN: các láng giềng có trọng số bằng nhau (`uniform`).
- WKNN: trọng số nghịch đảo khoảng cách trong không gian RSSI đã chuẩn hóa (`distance`).
  Trường hợp khoảng cách RSSI bằng 0 được scikit-learn xử lý riêng.
- Giữ nguyên sentinel -100 như baseline cũ; không giả định nó là phép đo tín hiệu vật lý.
- k không đủ số mẫu Train được ghi là bỏ qua, không âm thầm đổi k.
- Tiêu chí duy nhất là `MAE_distance_m` trên Validation; hòa thì chọn k nhỏ hơn.


In [ ]:
def json_ready(value):
    if isinstance(value, dict):
        return {str(key): json_ready(item) for key, item in value.items()}
    if isinstance(value, (list, tuple)):
        return [json_ready(item) for item in value]
    if isinstance(value, np.generic):
        return value.item()
    if isinstance(value, Path):
        return str(value)
    return value


def stable_digest(value):
    return hashlib.sha256(json.dumps(json_ready(value), ensure_ascii=False,
                                     sort_keys=True, allow_nan=False).encode()).hexdigest()


def frame_digest(frame):
    columns = ["Floor", "ID_RP", "Row_key", *XY_COLS, *AP_COLS]
    ordered = frame[columns].sort_values("Row_key")
    return hashlib.sha256(ordered.to_csv(index=False, float_format="%.17g").encode()).hexdigest()


def current_protocol():
    return {"run_id": RUN_ID, "config": copy.deepcopy(CONFIG), "AP_columns": AP_COLS.copy(),
            "zip_sha256": ZIP_SHA256, "libraries": LIBRARY_VERSIONS.copy(),
            "split_content_sha256": {
                floor: {name: frame_digest(frame) for name, frame in parts.items()}
                for floor, parts in splits.items()}}


VALIDATION_PROTOCOL_SIGNATURE = stable_digest(current_protocol())

def fit_real_baseline(train, k, weights):
    if len(train) < k:
        raise ValueError(f"Train có {len(train)} mẫu, ít hơn k={k}.")
    pipeline = Pipeline([
        ("scale", StandardScaler()),
        ("knn", KNeighborsRegressor(n_neighbors=k, weights=weights,
                                    algorithm="brute", n_jobs=CONFIG["n_jobs"])),
    ])
    with threadpool_limits(limits=CONFIG["n_jobs"]):
        pipeline.fit(train[AP_COLS], train[XY_COLS])
    # Bằng chứng scaler fit đúng tập, không có Validation/Test trong bước fit.
    np.testing.assert_allclose(pipeline.named_steps["scale"].mean_,
                               train[AP_COLS].mean().to_numpy(), rtol=1e-10, atol=1e-10)
    assert pipeline.named_steps["scale"].n_samples_seen_ == len(train)
    return pipeline


def predict_with_metrics(model, query):
    with threadpool_limits(limits=CONFIG["n_jobs"]):
        pred = model.predict(query[AP_COLS])
    metrics, errors = localization_metrics(query[XY_COLS], pred, query["RP_key"])
    details = query[["Floor", "ID_RP", "RP_key", "Row_key", "X", "Y"]].copy()
    details["Pred_X"], details["Pred_Y"] = pred[:, 0], pred[:, 1]
    details["Error_m"] = errors
    return metrics, details


validation_rows, skipped_rows, validation_prediction_parts = [], [], []
validation_models = {}
for floor, parts in splits.items():
    train, validation = parts["Train"], parts["Validation"]
    for method, weights in CONFIG["methods"].items():
        for k in sorted(set(CONFIG["k_candidates"])):
            if k > len(train):
                skipped_rows.append({"Floor": floor, "Method": method, "K": k,
                                     "Reason": "k lớn hơn số mẫu Train"})
                continue
            model = fit_real_baseline(train, k, weights)
            metrics, details = predict_with_metrics(model, validation)
            validation_models[floor, method, k] = model
            validation_rows.append({"Floor": floor, "Method": method, "K": k,
                                    "Weights": weights, "Fit_samples": len(train), **metrics})
            validation_prediction_parts.append(details.assign(Method=method, K=k, Split="Validation"))
validation_results_df = pd.DataFrame(validation_rows)
skipped_candidates_df = pd.DataFrame(skipped_rows)
expected_groups = len(CONFIG["floors"]) * len(CONFIG["methods"])
if validation_results_df.empty or validation_results_df.groupby(["Floor", "Method"]).ngroups != expected_groups:
    raise ValueError("Có tầng/phương pháp không có k hợp lệ; xem số mẫu và k_candidates.")
validation_predictions_df = pd.concat(validation_prediction_parts, ignore_index=True)
if stable_digest(current_protocol()) != VALIDATION_PROTOCOL_SIGNATURE:
    raise RuntimeError("Dữ liệu/cấu hình đổi trong lúc chạy Validation.")
VALIDATION_RESULTS_SIGNATURE = stable_digest(validation_results_df.to_dict(orient="records"))
display(validation_results_df.round(4))
if len(skipped_candidates_df):
    display(skipped_candidates_df)


## 11. Khóa lựa chọn trước khi xem Test

Giữ cả hai baseline đã đăng ký trước; mỗi phương pháp lấy k tốt nhất trên Validation.
Phương pháp ưu tiên cũng được chọn bằng Validation, không theo kết quả Test.
Hồ sơ khóa gồm cấu hình, hash ZIP, hash nội dung split và tham số đã chọn.
Nếu thay cấu hình/dữ liệu sau đó, phải chạy lại quy trình lựa chọn, không sửa kết quả Test để chọn k.


In [ ]:
TEST_ARTIFACT_NAMES = ("test_results.csv", "test_predictions.csv", "test_error_cdf.png", "test_error_map.png")


def invalidate_test_results():
    # Chỉ xóa output Test do notebook này tạo trong đúng thư mục lần chạy.
    global TEST_EXECUTED, TEST_SIGNATURE, test_results_df, test_predictions_df, final_models
    TEST_EXECUTED, TEST_SIGNATURE = False, None
    test_results_df, test_predictions_df, final_models = pd.DataFrame(), pd.DataFrame(), {}
    for name in TEST_ARTIFACT_NAMES:
        (OUTPUT_DIR / name).unlink(missing_ok=True)


def verify_current_lock(require_test=False):
    live_lock = {**LOCKED_SELECTION, "protocol": current_protocol()}
    if stable_digest(live_lock) != LOCK_SIGNATURE:
        raise RuntimeError("Cấu hình/dữ liệu/khóa lựa chọn đã thay đổi. Chạy lại quy trình từ đầu.")
    if stable_digest(validation_results_df.to_dict(orient="records")) != VALIDATION_RESULTS_SIGNATURE:
        raise RuntimeError("Bảng Validation đã bị đổi; chạy lại đánh giá trước khi xuất.")
    if stable_digest(selected_configs_df.to_dict(orient="records")) != stable_digest(LOCKED_SELECTION["selected_models"]):
        raise RuntimeError("Bảng cấu hình đã chọn không khớp khóa lựa chọn.")
    if require_test and (not TEST_EXECUTED or TEST_SIGNATURE != LOCK_SIGNATURE):
        raise RuntimeError("Kết quả Test không thuộc lựa chọn hiện hành; không xuất kết quả cũ.")


if stable_digest(current_protocol()) != VALIDATION_PROTOCOL_SIGNATURE:
    raise RuntimeError("Validation thuộc cấu hình/split cũ. Chạy lại mục 10 trước khi khóa.")
if stable_digest(validation_results_df.to_dict(orient="records")) != VALIDATION_RESULTS_SIGNATURE:
    raise RuntimeError("Bảng Validation đã thay đổi; không khóa kết quả chỉnh tay.")

# Khóa lại lựa chọn làm mất hiệu lực mọi kết quả Test từ lựa chọn trước.
invalidate_test_results()
for name in ("validation_by_k.png", "validation_selected_metrics.png"):
    (OUTPUT_DIR / name).unlink(missing_ok=True)


metric = CONFIG["selection_metric"]
if metric != "MAE_distance_m":
    raise ValueError("Giai đoạn 1 dùng MAE_distance_m nhất quán làm tiêu chí chính.")
selected_configs_df = (
    validation_results_df.sort_values(["Floor", "Method", metric, "K"], kind="stable")
    .groupby(["Floor", "Method"], sort=False).head(1).reset_index(drop=True)
)
primary_indices = selected_configs_df.groupby("Floor")[metric].idxmin()
primary_method_by_floor = selected_configs_df.loc[primary_indices].set_index("Floor")["Method"].to_dict()
LOCKED_SELECTION = {
    "protocol": current_protocol(),
    "selected_models": selected_configs_df.to_dict(orient="records"),
    "primary_method_by_floor": primary_method_by_floor.copy(),
    "selection_source": "Validation only",
}
LOCK_SIGNATURE = stable_digest(LOCKED_SELECTION)
selection_path = OUTPUT_DIR / "selection_lock.json"
selection_path.write_text(json.dumps(json_ready({**LOCKED_SELECTION, "signature": LOCK_SIGNATURE}),
                                     ensure_ascii=False, indent=2, allow_nan=False), encoding="utf-8")
display(selected_configs_df.round(4))
print("Phương pháp ưu tiên theo Validation:", primary_method_by_floor)
print("Đã lưu khóa lựa chọn:", selection_path)


## 12. Biểu đồ Validation

Đường theo k dùng để chọn trên Validation; biểu đồ cột dùng cùng công thức khoảng cách.
Không suy ra phương pháp tốt hơn nói chung chỉ từ một split/seed.


In [ ]:
fig, axes = plt.subplots(1, len(CONFIG["floors"]), figsize=(15, 4), squeeze=False)
for ax, floor in zip(axes[0], CONFIG["floors"]):
    for method in CONFIG["methods"]:
        part = validation_results_df.query("Floor == @floor and Method == @method").sort_values("K")
        ax.plot(part.K, part.MAE_distance_m, marker="o", label=method)
    ax.set(title=f"{floor} — Validation", xlabel="k", ylabel="Khoảng cách trung bình (m)")
    ax.set_xticks(sorted(validation_results_df.K.unique()))
    ax.grid(alpha=0.25)
    ax.legend()
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "validation_by_k.png", dpi=160, bbox_inches="tight")
plt.show()
plt.close(fig)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, column, title in zip(axes, ["MAE_distance_m", "RMSE_distance_m"],
                            ["Khoảng cách trung bình — Validation", "RMSE khoảng cách — Validation"]):
    selected_configs_df.pivot(index="Floor", columns="Method", values=column).plot.bar(ax=ax, rot=0)
    ax.set(title=title, ylabel="m", xlabel="Tầng")
    ax.grid(axis="y", alpha=0.2)
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "validation_selected_metrics.png", dpi=160, bbox_inches="tight")
plt.show()
plt.close(fig)


## 13. Đánh giá Test với cấu hình đã khóa

Mặc định bỏ qua (`RUN_FINAL_TEST=False` ngay đầu cell code bên dưới). Khi quy trình đã khóa,
đổi công tắc tại đây thành `True` rồi chạy cell này và các cell sau.
Không chạy lại mục 2 chỉ để bật Test. Không thay k/tiền xử lý dựa vào Test.

Nếu `refit_train_and_validation=True`, mỗi baseline fit lại scaler và KNN trên Train+Validation
trước Test; đây là bước fit cuối với tham số đã chọn, không phải tối ưu bằng Test.
Hai phương pháp đều được đăng ký trước; cột `Primary_on_validation` chỉ rõ lựa chọn từ Validation.
Không tự chọn phương pháp thắng từ bảng Test.


In [ ]:
RUN_FINAL_TEST = False  # chỉ bật sau khi đã khóa cấu hình ở mục 11
invalidate_test_results()
if not RUN_FINAL_TEST:
    print("CHƯA CHẠY TEST. Kết quả hiện có chỉ là Validation với dữ liệu đã nạp.")
else:
    verify_current_lock()
    test_rows, test_details = [], []
    for selected in LOCKED_SELECTION["selected_models"]:
        floor, method, k = selected["Floor"], selected["Method"], int(selected["K"])
        parts = splits[floor]
        fit_frame = (pd.concat([parts["Train"], parts["Validation"]], ignore_index=True)
                     if CONFIG["refit_train_and_validation"] else parts["Train"])
        model = fit_real_baseline(fit_frame, k, selected["Weights"])
        metrics, details = predict_with_metrics(model, parts["Test"])
        final_models[floor, method] = model
        is_primary = method == LOCKED_SELECTION["primary_method_by_floor"][floor]
        test_rows.append({"Floor": floor, "Method": method, "K": k,
                          "Primary_on_validation": is_primary,
                          "Fit_scope": "Train+Validation" if CONFIG["refit_train_and_validation"] else "Train",
                          "Fit_samples": len(fit_frame), **metrics})
        test_details.append(details.assign(Method=method, K=k, Split="Test"))
    test_results_df = pd.DataFrame(test_rows)
    test_predictions_df = pd.concat(test_details, ignore_index=True)
    TEST_EXECUTED = True
    TEST_SIGNATURE = LOCK_SIGNATURE
    display(test_results_df.round(4))
    print("Đã đánh giá hai baseline đã đăng ký; không có bước chọn tham số bằng Test.")


## 14. Trực quan hóa sai số Test

Chỉ có hình khi Test thật sự đã chạy. CDF cho biết tỷ lệ mẫu có lỗi không vượt một khoảng cách.
Bản đồ màu thể hiện lỗi trung bình theo RP của phương pháp đã chọn trên Validation.


In [ ]:
if not TEST_EXECUTED:
    print("Bỏ qua biểu đồ Test vì Test chưa chạy; không dựng số liệu thay thế.")
else:
    verify_current_lock(require_test=True)
    fig, axes = plt.subplots(1, len(CONFIG["floors"]), figsize=(15, 4), squeeze=False)
    for ax, floor in zip(axes[0], CONFIG["floors"]):
        for method in CONFIG["methods"]:
            values = test_predictions_df.query("Floor == @floor and Method == @method").Error_m.to_numpy()
            ordered = np.sort(values)
            ax.step(ordered, np.arange(1, len(ordered) + 1) / len(ordered), where="post", label=method)
        ax.set(title=f"{floor} — CDF Test", xlabel="Sai số định vị (m)", ylabel="Tỷ lệ mẫu", ylim=(0, 1.02))
        ax.grid(alpha=0.25)
        ax.legend()
    fig.tight_layout()
    fig.savefig(OUTPUT_DIR / "test_error_cdf.png", dpi=160, bbox_inches="tight")
    plt.show()
    plt.close(fig)

    fig, axes = plt.subplots(1, len(CONFIG["floors"]), figsize=(15, 4), squeeze=False)
    for ax, floor in zip(axes[0], CONFIG["floors"]):
        method = LOCKED_SELECTION["primary_method_by_floor"][floor]
        part = test_predictions_df.query("Floor == @floor and Method == @method")
        grouped = part.groupby("RP_key", as_index=False).agg(X=("X", "mean"), Y=("Y", "mean"),
                                                                  Error_m=("Error_m", "mean"))
        points = ax.scatter(grouped.X, grouped.Y, c=grouped.Error_m, cmap="viridis", s=35)
        fig.colorbar(points, ax=ax, label="Lỗi trung bình/RP (m)")
        ax.set(title=f"{floor} — {method} chọn từ Validation", xlabel="X (m)", ylabel="Y (m)")
        ax.set_aspect("equal", adjustable="box")
    fig.tight_layout()
    fig.savefig(OUTPUT_DIR / "test_error_map.png", dpi=160, bbox_inches="tight")
    plt.show()
    plt.close(fig)


## 15. Xuất bảng, manifest và hồ sơ lần chạy

Chỉ xuất bảng Test khi đã chạy Test. Mỗi lần chạy có thư mục riêng để không trộn output cũ.
`run_metadata.json` ghi rõ nguồn, phiên bản, công thức và trạng thái kiểm thử.
ZIP kết quả chứa bảng/biểu đồ/metadata, **không tự đóng gói dataset nguồn**.
Tải kết quả từ thanh Files của Colab; lưu sang Drive nếu cần giữ qua lần reset runtime.


In [ ]:
verify_current_lock(require_test=TEST_EXECUTED)
tables = {
    "workbook_inventory": workbook_inventory,
    "input_audit": audit_df,
    "split_summary": split_summary_df,
    "split_manifest": split_manifest_df,
    "split_overlap_audit": overlap_audit_df,
    "train_ap_audit": train_ap_audit_df,
    "validation_candidates": validation_results_df,
    "validation_selected": selected_configs_df,
    "validation_predictions": validation_predictions_df,
}
if len(skipped_candidates_df):
    tables["skipped_candidates"] = skipped_candidates_df
if TEST_EXECUTED:
    tables.update(test_results=test_results_df, test_predictions=test_predictions_df)
for name, frame in tables.items():
    frame.to_csv(OUTPUT_DIR / f"{name}.csv", index=False, encoding="utf-8-sig")

run_metadata = {
    "run_id": RUN_ID, "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "stage": "Stage 1: real-data KNN/WKNN only", "config": CONFIG,
    "python": sys.version, "libraries": LIBRARY_VERSIONS,
    "source_zip_name": DATA_ZIP.name, "source_zip_sha256": ZIP_SHA256,
    "source_sheets": {floor: f"DLULIB_Train_{floor}.xlsx::Train_{floor}_2" for floor in CONFIG["floors"]},
    "selection_signature": LOCK_SIGNATURE, "test_executed": TEST_EXECUTED,
    "test_role": "Internal RP holdout from Train workbooks; previously explored in original notebook",
    "test_is_unseen_research_holdout": False,
    "rss_reconstruction_evaluated": False,
    "metrics": {"MAE_distance_m": "mean(norm(pred_xy - true_xy, axis=1))",
                "RMSE_distance_m": "sqrt(mean(norm(pred_xy - true_xy, axis=1)**2))",
                "MAE_RP_macro_m": "mean over RP of mean distance error within RP"},
    "limitations": ["Floor assumed known", "Coordinate unit m requires dataset documentation",
                    "No session/device/time metadata audit", "Random RP split, not spatial block split",
                    "No cGAN or interpolation training/evaluation in Stage 1"],
}
(OUTPUT_DIR / "run_metadata.json").write_text(
    json.dumps(json_ready(run_metadata), ensure_ascii=False, indent=2, allow_nan=False), encoding="utf-8")

result_zip = OUTPUT_DIR.parent / f"{RUN_ID}_results.zip"
with zipfile.ZipFile(result_zip, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    # Danh sách tệp hiện hành rõ ràng; không gom nhầm kết quả Test từ lần chạy cell trước.
    artifact_names = [f"{name}.csv" for name in tables]
    artifact_names += ["selection_lock.json", "run_metadata.json", "split_map.png",
                       "train_rssi_audit.png", "validation_by_k.png", "validation_selected_metrics.png"]
    if TEST_EXECUTED:
        artifact_names += ["test_error_cdf.png", "test_error_map.png"]
    for name in artifact_names:
        path = OUTPUT_DIR / name
        if path.is_file():
            archive.write(path, arcname=name)
print("Bảng và hình:", OUTPUT_DIR)
print("ZIP kết quả:", result_zip)
print("Trạng thái Test:", "Đã chạy" if TEST_EXECUTED else "CHƯA CHẠY")
print("Chưa đánh giá RSSI tái tạo hoặc hiệu quả tăng cường cGAN/nội suy ở Giai đoạn 1.")


## 16. Cách đọc kết quả và phát triển tiếp

1. Mở `https://colab.research.google.com/` → **File/Tệp → Upload notebook/Tải notebook lên**
   → chọn `KLTN_CGAN_Experiment.ipynb` trên máy. Chưa cần liên kết GitHub đã publish.
2. Chạy từ đầu, chọn upload ZIP hoặc Drive tại mục 4. Nếu audit báo lỗi, kiểm tra nguyên nhân;
   không tự đổi mã RSSI hoặc xóa dòng để đạt kết quả tốt hơn.
3. Xem `input_audit`, `split_overlap_audit`, bản đồ split và các bảng Validation.
   Không bật Test trong lúc còn khảo sát phương pháp.
4. Khi giao thức đã khóa, bật `RUN_FINAL_TEST=True` ngay cell mục 13 và chạy mục 13–15.
   Không dùng bảng Test để chọn k/method hoặc lặp thay split nhằm cải thiện kết quả.
5. Tải ZIP kết quả từ Files; hồ sơ ghi rõ Test đã chạy hay chưa.

**Chưa thể kết luận:** dữ liệu sinh tốt hơn, cGAN hội tụ, hoặc tăng cường cải thiện định vị.
Stage 1 chỉ tạo baseline có công thức và giao thức kiểm tra rõ ràng.
Hàm metric RSSI chỉ được dùng khi có cặp đối chứng đúng nghĩa ở giai đoạn sau;
không ghép tùy ý fingerprint GAN ngẫu nhiên với phép đo thật rồi coi là lỗi tái tạo.

Các bước tiếp theo sau phê duyệt: IDW, đánh giá RSSI tại RP giữ lại, cGAN tại cùng điểm sinh,
ngân sách tăng cường giống nhau, nhiều seed huấn luyện và đánh giá theo vùng không gian.
Notebook gốc và kiến trúc cGAN được giữ nguyên trong giai đoạn này.
